#### Here we will use a technique called bag of words, it is a matrix having count words w.r.t to a document

####

Then on that resultant matrix we apply SVD
$A=U \sum V^T$


In [22]:
# load a PDF file
import requests
import os

url = 'https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf'

# Get PDF docuement path
file_name = 'topics_6.pdf'
output_path = f'{file_name}'

if not os.path.exists(output_path):
    print(f'[INFO] file doesnt exist, downloading...')
    try:
        response = requests.get(url)

        # check if request was successful
        if response.status_code == 200:
            # open the file and save it
            with open(output_path, 'wb') as f:
                f.write(response.content)

            print(f"[INFO] file has been downloaded and saved as {file_name}");
        else:
            print(f'[INFO] Failed to download the file. Status code: {response.status_code}')

    except Exception as e:
        print(f"An error occurred: {e}")

else:
    print(f'File {output_path} exists')



File topics_6.pdf exists


In [23]:
import torch
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm

In [24]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\d1990\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\d1990\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [26]:
stop_words = set(stopwords.words('english'))

In [27]:
def preprocess(docs):
    result = []
    stop_words = set(stopwords.words('english'))
    for text in docs:
        #convert to lowercase, then replace non alpha chars with empty
        text = re.sub(r'[^a-z\s]', '', text.lower())
        tokens = word_tokenize(text)
        # remove stopwords
        filtered_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
        result.append(filtered_tokens)
    return result

In [28]:
import fitz #pymupdf
from tqdm.auto import tqdm

def open_and_read_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = preprocess([page.get_text()])[0]
        if (len(text) > 10):
            pages_and_texts.append({
                "text": text,
                "raw_text": page.get_text(),
                "page_number": len(pages_and_texts) + 1
                }) 
        
    return pages_and_texts


pages_and_texts = open_and_read_pdf(output_path)

6it [00:00, 106.17it/s]


In [29]:
pages_and_texts[:5]

[{'text': ['cybersecurity',
   'practice',
   'protecting',
   'systems',
   'networks',
   'data',
   'digital',
   'attacks',
   'often',
   'aimed',
   'accessing',
   'changing',
   'destroying',
   'sensitive',
   'information',
   'effective',
   'cybersecurity',
   'relies',
   'people',
   'processes',
   'technology',
   'working',
   'together',
   'key',
   'principles',
   'cia',
   'triad',
   'confidentiality',
   'ensuring',
   'information',
   'remains',
   'secret',
   'authorized',
   'users',
   'access',
   'integrity',
   'protecting',
   'data',
   'altered',
   'manipulated',
   'deleted',
   'unauthorized',
   'parties',
   'availability',
   'making',
   'information',
   'systems',
   'accessible',
   'authorized',
   'users',
   'needed',
   'common',
   'threats',
   'malware',
   'malicious',
   'software',
   'like',
   'viruses',
   'trojans',
   'ransomware',
   'designed',
   'damage',
   'gain',
   'unauthorized',
   'access',
   'systems',
   'phishi

In [30]:
#build a vocab
vocab = set()

# Loop through num_sentence_chunk_size and split sentences into chunks
for item in tqdm(pages_and_texts):
    topic = item["text"]
    words = topic
    for w in words:
        if (len(w.strip()) > 0):
            vocab.add(w)

100%|██████████| 6/6 [00:00<00:00, 6004.73it/s]


In [31]:
vocab = sorted(list(vocab))
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [32]:
len(vocab)

534

In [33]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [34]:
# bag of words
bow = torch.zeros(len(pages_and_texts), len(vocab), dtype=torch.float, device=device)

In [35]:
def bag_of_words():
    # we could use tokenization, but would use simple technique
    for i, doc in enumerate(pages_and_texts):
        for word in doc["text"]:
            if word in word_to_idx:
                bow[i, word_to_idx[word]] += 1

bag_of_words()


In [36]:
bow.shape

torch.Size([6, 534])

In [37]:
#now do SVD on bow
U, S, V_t = torch.linalg.svd(bow)

In [38]:
U.shape, S.shape, V_t.shape

(torch.Size([6, 6]), torch.Size([6]), torch.Size([534, 534]))

In [39]:
num_topics = 2
#taking top 5 axis or directions based on decreasing order of sigular values, unit vectors
# order defines significance of each axis or direction
U = U[:, :num_topics] #five important from C(bow)
S = torch.diag(S[:num_topics])
V_t = V_t[:num_topics, :] # five important from C(bow^T) or rowspace of bow

In [40]:
doc_topics = U @ S #scaling variance in the direction of U by amount S
word_topics = V_t.T @ S # similarly for row space

In [41]:
word_topics.shape, doc_topics.shape

(torch.Size([534, 2]), torch.Size([6, 2]))

In [42]:
#display top words of each topic
def display_words(num_words = 15):
    print("\n Top words per topic")
    for idx in range(word_topics.size(1)):
        words = word_topics[:, idx]
        top_indices = torch.argsort(words, descending=True)[:num_words]
        top_words = [vocab[i] for i in top_indices]
        print(f"Topic {idx + 1}: {', '.join(top_words)}")


display_words()


 Top words per topic
Topic 1: central, personal, writing, works, great, wellknown, lottery, gift, collection, adds, eg, jackson, particularly, nonfiction, themed
Topic 2: internet, access, tcpip, web, email, key, social, data, systems, passwords, users, research, invented, arpanet, platforms


In [43]:
def format_doc_topics(idx, top_topics):
    
    print(f"\nDoc {idx + 1}:")
    for i, topic in enumerate(top_topics):
        print(f" Topic: {i} \n\t{topic.strip().replace('\n', '')}")


#display top topics of each document (uses the helper above)
def display_topics(num_topics: int = 3):
    print("Top topics per doc")
    for idx in range(doc_topics.size(1)):
        topics = doc_topics[:, idx]
        top_indices = torch.argsort(topics, descending=True)[:num_topics]
        top_topics = [pages_and_texts[i]["raw_text"][:125] for i in top_indices]
        format_doc_topics(idx, top_topics)


display_topics()

Top topics per doc

Doc 1:
 Topic: 0 
	This topic is a great opportunity for creativity. You can select existing works you love or include some of your own writi
 Topic: 1 
	A balanced diet provides the necessary energy and nutrients for optimal physical and mental well-being. It is not about ri
 Topic: 2 
	Stargazing is a rewarding hobby that allows you to connect with the wider universe. You can start with minimal equipment.

Doc 2:
 Topic: 0 
	The Internet evolved from a military project into a global communication backbone through several key technological advanc
 Topic: 1 
	Cybersecurity is the practice of protecting systems, networks, and data from digital attacks, which are often aimed at acces
 Topic: 2 
	A balanced diet provides the necessary energy and nutrients for optimal physical and mental well-being. It is not about ri
